# Full pipeline runner

Runs the five modelling notebooks in dependency order and lets you change the scenario
assumptions (renovation rate, demolition rate, HP adoption, COP, ...) from one place instead of
editing each notebook.

**Pipeline (dependency order, not alphabetical):**

1. `data_preparation.ipynb` – country-year panel (`panel_transformed.csv`)
2. `floor_area_panel.ipynb` – floor-area / thermal-index panel (`floor_panel_2010_2060.csv`)
3. `heat_pump_ols.ipynb` – heat-pump electricity demand (`hist_hp_heat_demand.csv`,
   `Results/hp_demand_proj.csv`)
4. `buildings_rest_ols.ipynb` – buildings electricity demand excl. HP & data centers
   (`Results/building_demand_proj.csv`) – needs step 3's output
5. `praesentation_grafiken.ipynb` – exports all figures to `figures/`

`buildings_rest_ols.ipynb` has to run after `heat_pump_ols.ipynb` even though the filename
sorts earlier – it reads `hist_hp_heat_demand.csv`, which step 3 produces.

**Not produced by any of these five, has to exist already:**
- `Results/data_centers_demand.xlsx` (separate data-center model)
- `Data - raw/chdd_data/output_csv/hdd_cdd_scenarios_2025_2060.csv` (from
  `chdd_calculation_scenarios.ipynb`)

**How the tuning works:** all five notebooks import their constants (renovation rate,
demolition rate, HP adoption curve, COP, ...) from `scenario_config.py` instead of hardcoding
them. This notebook regenerates `scenario_config.py` from the `PARAMS` dict below before each
run, so there's one place to change a value. The notebooks themselves aren't touched – running
them standalone still works, they just pick up whatever is currently in `scenario_config.py`.

Note: each run overwrites `scenario_config.py`. A hand-tuned value that should survive the next
orchestrator run needs to go into `PARAMS` below, not directly into `scenario_config.py`.

## 1. Setup & preflight checks

In [1]:
import time
import shutil
from pathlib import Path
from datetime import datetime

import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

CODE_DIR = Path.cwd()
assert (CODE_DIR / "data_preparation.ipynb").exists(), (
    "Run this notebook from the 'Code' project folder (its own directory) — "
    f"'data_preparation.ipynb' was not found in {CODE_DIR}"
)

REQUIRED_EXTERNAL_FILES = [
    CODE_DIR / "Results" / "data_centers_demand.xlsx",
    CODE_DIR / "Data - raw" / "chdd_data" / "output_csv" / "hdd_cdd_scenarios_2025_2060.csv",
]
missing = [p for p in REQUIRED_EXTERNAL_FILES if not p.exists()]
if missing:
    print("WARNING - external prerequisite files not found (produced outside this pipeline):")
    for p in missing:
        print("  -", p)
    print("buildings_rest_ols.ipynb / praesentation_grafiken.ipynb will fail without these.")
else:
    print("All external prerequisite files found.")

All external prerequisite files found.


## 2. Scenario parameters – tune here

Control panel for the whole pipeline. Every value below is read by one or more of the five
notebooks via `scenario_config.py`. Change a value, then run all cells below to regenerate
`scenario_config.py` and execute the pipeline with the new assumptions.

In [ ]:
PARAMS = {

    # ---- Building stock / floor area (floor_area_panel.ipynb) --------------------------

    # Heat-demand intensity by construction cohort, kWh/m2a (calibrate against TABULA).
    "BASE_INTENSITY": {
        "0-1945":    250,
        "1946-1969": 230,
        "1970-1979": 170,
        "1980-1989": 135,
        "1990-1999": 110,
        "2000-2010":  85,
        "2011-2020":  50,
        "post-2020":  35,   # NZEB standard, mandatory for new builds from 2021 (EPBD Art. 9)
    },

    # Annual demolition hazard by cohort. EU-wide demolition is typically 0.1-0.3%/yr of the
    # stock; older/pre-1990 cohorts justify the higher end, newer cohorts ~0.
    "DEMOLITION_RATE": {
        "0-1945":    0.0030,
        "1946-1969": 0.0020,
        "1970-1979": 0.0010,
        "1980-1989": 0.0005,
        "1990-1999": 0.0000,
        "2000-2010": 0.0000,
        "2011-2020": 0.0000,
        "post-2020": 0.0000,
    },

    "RENOVATION_DEPTH": 0.35,       # average relative heat-demand reduction achieved per renovation
    "RENOVATION_START": 0.010,      # EU average energy-renovation rate in 2025 (~1%/yr)
    "HIST_RENOVATION_RATE": 0.00,   # or 0.01,   

    # Scenario -> annual renovation rate reached by 2030 (held constant thereafter).
    # KKO/KVE/VWE/VNI are linked 1:1 to the SSP1/SSP2/SSP5/SSP3 climate paths below.
    "SCENARIO_RENOVATION": {
        "KKO": 0.020,   # SSP1 - EU Renovation Wave target met (~2%/yr, EPBD ambition)
        "KVE": 0.015,   # SSP2 - moderate acceleration
        "VWE": 0.015,   # SSP5
        "VNI": 0.010,   # SSP3 - status quo, no acceleration
    },

    "EPSILON_DEFAULT": 0.25,          # fallback income elasticity of floor area per capita
    "EPSILON_BOUNDS": (0.05, 0.60),   # plausibility range for the per-country estimated elasticity
    "PERMIT_LAG": 1,                  # years between a building permit and floor area entering the stock

    # ---- Heat pump adoption & COP (heat_pump_ols.ipynb, buildings_rest_ols.ipynb) -------

    # HP-adoption S-curve: year the max share is reached, and the target fraction of that max.
    "HP_SCENARIOS": {
        "KKO": {"target_year": 2050, "target_fraction": 0.9},
        "KVE": {"target_year": 2055, "target_fraction": 0.9},
        "VWE": {"target_year": 2060, "target_fraction": 0.9},
        "VNI": {"target_year": 2060, "target_fraction": 0.75},
    },

    "USE_DYNAMIC_COP": True,   # False -> constant per-country COP; True -> coupled to thermal_index
    "COP_ALPHA": 0.3,          # coupling strength of COP to thermal_index (0 = no effect)
    "THERMAL_REF": 150,        # kWh/m2a at which COP = COP_base (anchor point of the formula)
    "COP_MAX": 6.5,            # upper cap on COP
    "SERVICE_MULTIPLIER": 1,   # HP demand multiplier for the service sector on top of residential

    # Country base COP - climate-zone / heat-source temperature differences
    # (Nouvel et al., 2015): Nordic=3.3, West/Central=4.0, Eastern=3.5, Southern=4.3
    "COP_MAP": {
        "Austria": 4.0, "Belgium": 4.0, "Bulgaria": 3.5, "Croatia": 4.3,
        "Czechia": 3.5, "Denmark": 3.3, "Estonia": 3.3, "Finland": 3.3,
        "France": 4.0,  "Germany": 4.0, "Greece": 4.3,  "Hungary": 3.5,
        "Ireland": 4.0, "Italy": 4.0,   "Latvia": 3.3,  "Lithuania": 3.3,
        "Luxembourg": 4.0, "Netherlands": 4.0, "Norway": 3.3, "Poland": 3.5,
        "Portugal": 4.3,   "Romania": 3.5,    "Slovakia": 3.5, "Slovenia": 4.3,
        "Spain": 4.3,      "Sweden": 3.3,
    },

    # ---- Climate / scenario wiring (all three model notebooks) -------------------------

    "USE_HDD_CDD_SCENARIOS": True,   # False -> single RCP4.5 pathway; True -> 4 SSP-linked paths
    "CLIMATE_SCENARIO_MAP": {"KKO": "SSP1", "KVE": "SSP2", "VWE": "SSP5", "VNI": "SSP3"},
    "FLOOR_BASELINE_SCENARIO": "KKO",   # scenario used for the single-pathway (non-scenario) projection
}

## 3. Optional scenario presets

Ready-made overrides for the usual sensitivity checks (useful for the thesis' robustness
section). Pick one via `ACTIVE_PRESET`, or leave `"baseline"` to run `PARAMS` unmodified. Add
more the same way – each preset is a partial override merged onto `PARAMS`.

In [3]:
"""PRESETS = {
    "baseline": {},  # PARAMS above, unmodified

    "accelerated_renovation": {
        # Fit-for-55 / stronger Renovation Wave push across all scenarios
        "SCENARIO_RENOVATION": {"KKO": 0.030, "KVE": 0.022, "VWE": 0.022, "VNI": 0.015},
        "RENOVATION_DEPTH": 0.45,
    },

    "slow_transition": {
        # policy-inertia case: renovation stays close to today's rate, demolition halves
        "SCENARIO_RENOVATION": {"KKO": 0.012, "KVE": 0.010, "VWE": 0.010, "VNI": 0.007},
        "DEMOLITION_RATE": {k: v * 0.5 for k, v in PARAMS["DEMOLITION_RATE"].items()},
    },

    "high_demolition": {
        # aggressive stock replacement / new-build-heavy case
        "DEMOLITION_RATE": {k: v * 2 for k, v in PARAMS["DEMOLITION_RATE"].items()},
    },

    "constant_cop": {
        # switch off the thermal_index-coupled COP model (old constant-per-country behaviour)
        "USE_DYNAMIC_COP": False,
    },

    "single_climate_path": {
        # use one RCP4.5 pathway everywhere instead of 4 SSP-linked climate scenarios
        "USE_HDD_CDD_SCENARIOS": False,
    },
}

ACTIVE_PRESET = "baseline"   # <- change this to switch scenario bundles

if ACTIVE_PRESET != "baseline":
    overrides = PRESETS[ACTIVE_PRESET]
    PARAMS = {**PARAMS, **overrides}
    print(f"Applied preset '{ACTIVE_PRESET}': overrode {list(overrides)}")
else:
    print("Using baseline PARAMS (no preset applied).")"""

'PRESETS = {\n    "baseline": {},  # PARAMS above, unmodified\n\n    "accelerated_renovation": {\n        # Fit-for-55 / stronger Renovation Wave push across all scenarios\n        "SCENARIO_RENOVATION": {"KKO": 0.030, "KVE": 0.022, "VWE": 0.022, "VNI": 0.015},\n        "RENOVATION_DEPTH": 0.45,\n    },\n\n    "slow_transition": {\n        # policy-inertia case: renovation stays close to today\'s rate, demolition halves\n        "SCENARIO_RENOVATION": {"KKO": 0.012, "KVE": 0.010, "VWE": 0.010, "VNI": 0.007},\n        "DEMOLITION_RATE": {k: v * 0.5 for k, v in PARAMS["DEMOLITION_RATE"].items()},\n    },\n\n    "high_demolition": {\n        # aggressive stock replacement / new-build-heavy case\n        "DEMOLITION_RATE": {k: v * 2 for k, v in PARAMS["DEMOLITION_RATE"].items()},\n    },\n\n    "constant_cop": {\n        # switch off the thermal_index-coupled COP model (old constant-per-country behaviour)\n        "USE_DYNAMIC_COP": False,\n    },\n\n    "single_climate_path": {\n        #

## 4. Write `scenario_config.py` from `PARAMS`

In [4]:
def _render_value(v, indent=4):
    if isinstance(v, dict):
        pad = " " * indent
        inner = ",\n".join(f"{pad}{k!r}: {v2!r}" for k, v2 in v.items())
        return "{\n" + inner + f",\n" + " " * (indent - 4) + "}"
    return repr(v)

def generate_scenario_config(params: dict) -> str:
    lines = [
        '"""',
        "Central, single-source-of-truth scenario / model parameters for the demand pipeline.",
        "",
        f"AUTO-GENERATED by run_full_pipeline.ipynb on {datetime.now().isoformat(timespec='seconds')}.",
        "Edit PARAMS in that notebook and re-run it to regenerate this file - manual edits",
        "here will be overwritten on the next pipeline run.",
        '"""',
        "",
    ]
    for key, value in params.items():
        lines.append(f"{key} = {_render_value(value)}")
        lines.append("")
    return "\n".join(lines)

config_path = CODE_DIR / "scenario_config.py"
config_path.write_text(generate_scenario_config(PARAMS))
print(f"Wrote {config_path} ({len(PARAMS)} parameters).")

Wrote /Users/morteza/Library/CloudStorage/OneDrive-TechnischeUniversitätBerlin/Dokumente/2. WiIng/4. Masterarbeit SS26/Code/scenario_config.py (19 parameters).


## 5. Execution engine

In [5]:
RUN_DIR = CODE_DIR / "pipeline_runs" / datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Executed-notebook copies (with outputs) will be saved to: {RUN_DIR}")

def run_notebook(nb_filename: str, timeout: int = 1800) -> float:
    """Execute one of the pipeline notebooks in-place (fresh kernel, cwd = CODE_DIR) and
    save an executed copy with outputs to RUN_DIR. The original notebook file on disk is
    never modified. Raises CellExecutionError on failure."""
    src_path = CODE_DIR / nb_filename
    print(f"\n{'='*70}\nRunning {nb_filename} ...")
    t0 = time.time()
    nb = nbformat.read(src_path, as_version=4)
    kernel_name = nb.metadata.get("kernelspec", {}).get("name", "python3")
    client = NotebookClient(
        nb,
        timeout=timeout,
        kernel_name=kernel_name,
        resources={"metadata": {"path": str(CODE_DIR)}},
    )
    try:
        client.execute()
        elapsed = time.time() - t0
        print(f"{nb_filename}: OK in {elapsed:.1f}s")
        return elapsed
    except CellExecutionError as e:
        elapsed = time.time() - t0
        print(f"{nb_filename}: FAILED after {elapsed:.1f}s")
        raise
    finally:
        out_path = RUN_DIR / nb_filename.replace(".ipynb", "_executed.ipynb")
        nbformat.write(nb, out_path)
        print(f"  -> executed copy saved to {out_path}")

Executed-notebook copies (with outputs) will be saved to: /Users/morteza/Library/CloudStorage/OneDrive-TechnischeUniversitätBerlin/Dokumente/2. WiIng/4. Masterarbeit SS26/Code/pipeline_runs/20260902_163951


## 6. Run the full pipeline (dependency order)

In [6]:
PIPELINE = [
    "data_preparation.ipynb",
    "floor_area_panel.ipynb",
    "heat_pump_ols.ipynb",
    "buildings_rest_ols.ipynb",
    "praesentation_grafiken.ipynb",
]

results = {}
total_t0 = time.time()

for nb_name in PIPELINE:
    try:
        elapsed = run_notebook(nb_name)
        results[nb_name] = ("OK", elapsed)
    except Exception as e:
        results[nb_name] = ("FAILED", str(e)[:500])
        print(f"\nPipeline stopped: {nb_name} failed. Fix the error (see traceback above / "
              f"in the executed copy in {RUN_DIR}) and re-run this cell.")
        break
else:
    print(f"\nPipeline finished successfully in {time.time() - total_t0:.1f}s total.")

print("\nSummary:")
for nb_name in PIPELINE:
    status = results.get(nb_name, ("SKIPPED", None))
    print(f"  {status[0]:8s}  {nb_name}")


Running data_preparation.ipynb ...
data_preparation.ipynb: OK in 4.0s
  -> executed copy saved to /Users/morteza/Library/CloudStorage/OneDrive-TechnischeUniversitätBerlin/Dokumente/2. WiIng/4. Masterarbeit SS26/Code/pipeline_runs/20260902_163951/data_preparation_executed.ipynb

Running floor_area_panel.ipynb ...
floor_area_panel.ipynb: OK in 5.5s
  -> executed copy saved to /Users/morteza/Library/CloudStorage/OneDrive-TechnischeUniversitätBerlin/Dokumente/2. WiIng/4. Masterarbeit SS26/Code/pipeline_runs/20260902_163951/floor_area_panel_executed.ipynb

Running heat_pump_ols.ipynb ...
heat_pump_ols.ipynb: OK in 8.6s
  -> executed copy saved to /Users/morteza/Library/CloudStorage/OneDrive-TechnischeUniversitätBerlin/Dokumente/2. WiIng/4. Masterarbeit SS26/Code/pipeline_runs/20260902_163951/heat_pump_ols_executed.ipynb

Running buildings_rest_ols.ipynb ...
buildings_rest_ols.ipynb: OK in 4.7s
  -> executed copy saved to /Users/morteza/Library/CloudStorage/OneDrive-TechnischeUniversitä

## 7. Verify key outputs

In [7]:
expected_outputs = [
    CODE_DIR / "Data - raw" / "prepared_data" / "panel_transformed.csv",
    CODE_DIR / "Data - raw" / "prepared_data" / "floor_panel_2010_2060.csv",
    CODE_DIR / "Data - raw" / "prepared_data" / "hist_hp_heat_demand.csv",
    CODE_DIR / "Results" / "hp_demand_proj.csv",
    CODE_DIR / "Results" / "building_demand_proj.csv",
    CODE_DIR / "figures" / "fig_br_grid.png",
]

for p in expected_outputs:
    if p.exists():
        mtime = datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f"  OK    {p.relative_to(CODE_DIR)}  (updated {mtime})")
    else:
        print(f"  MISSING  {p.relative_to(CODE_DIR)}")

  OK    Data - raw/prepared_data/panel_transformed.csv  (updated 2026-09-02 16:39:54)
  OK    Data - raw/prepared_data/floor_panel_2010_2060.csv  (updated 2026-09-02 16:40:00)
  OK    Data - raw/prepared_data/hist_hp_heat_demand.csv  (updated 2026-09-02 16:40:02)
  OK    Results/hp_demand_proj.csv  (updated 2026-09-02 16:40:08)
  OK    Results/building_demand_proj.csv  (updated 2026-09-02 16:40:13)
  OK    figures/fig_br_grid.png  (updated 2026-09-02 16:40:17)
